# Instalación y configuración

In [1]:
%pip install -q "dask[complete]" gcsfs pyarrow fsspec plotly --root-user-action=ignore

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -U --force-reinstall --no-cache-dir \
  "numpy>=1.26,<1.27" \
  "pandas>=2.2,<3" \
  "dask[dataframe,distributed]" \
  "distributed"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 228.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 333.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 657.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 789.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 MB 335.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.6/806.6 kB 728.6 MB/s  0:00:00
  Attempting uninstall: sortedcontainers
    Found existing installation: sortedcontainers 2.4.0
    Uninstalling sortedcontainers-2.4.0:
      Successfully uninstalled sortedcontainers-2.4.0
  Attempting uninstall: pytz
    Found existing installation: pytz 2026.2
    Uninstalling pytz-2026.2:
      Successfully uninstalled pytz-2026.2
  Attempting uninstall: zipp
    Found existing installation: zipp 3.23.1
    Uninstalling zipp-3.23.1:
      Successfully uninstalled zipp-3.23.1
  Attempting uninstall: zict
    Found existing instal

In [3]:
import os
import gc
import subprocess

import dask.dataframe as dd
import pandas as pd
import numpy as np

### Rutas del proyecto en GCS

In [4]:
GCS_RAW_PATH = "gs://proyectobigdata2026/raw/restaurant_inspections.csv"

GCS_PROCESSED_DASK = "gs://proyectobigdata2026/processed/dask/restaurant_inspections_clean/"
GCS_OUTPUTS_DASK = "gs://proyectobigdata2026/results/outputs/dask/"
GCS_KPIS_DASK = "gs://proyectobigdata2026/results/kpis/dask/"

GCS_STORAGE_OPTIONS = {
    "token": "google_default"
}

In [5]:
!gcloud storage ls gs://proyectobigdata2026/raw/

gs://proyectobigdata2026/raw/

gs://proyectobigdata2026/raw/:
gs://proyectobigdata2026/raw/
gs://proyectobigdata2026/raw/restaurant_inspections.csv


In [6]:
dtype_config = {
    "camis": "object",
    "dba": "object",
    "boro": "object",
    "building": "object",
    "street": "object",
    "zipcode": "object",
    "phone": "object",
    "cuisine_description": "object",
    "inspection_date": "object",
    "action": "object",
    "violation_code": "object",
    "violation_description": "object",
    "critical_flag": "object",
    "score": "object",
    "grade": "object",
    "grade_date": "object",
    "record_date": "object",
    "inspection_type": "object",
    "latitude": "float64",
    "longitude": "float64",
    "community_board": "float64",
    "council_district": "float64",
    "census_tract": "float64",
    "bin": "float64",
    "bbl": "float64",
    "nta": "object",
    "location": "object"
}

df_raw = dd.read_csv(
    GCS_RAW_PATH,
    dtype=dtype_config,
    assume_missing=True,
    blocksize="32MB",
    storage_options=GCS_STORAGE_OPTIONS
)

print("Número de particiones:", df_raw.npartitions)
print("Columnas:", len(df_raw.columns))

df_raw.head()

Número de particiones: 4
Columnas: 27


,camis,dba,boro,building,street,zipcode,phone,cuisine_description,inspection_date,action,...,inspection_type,longitude,latitude,council_district,bin,community_board,nta,census_tract,bbl,location
0,50180205,SOHO PIZZA,Queens,42-27,35 AVENUE,11101,3472422217,<NA>,1900-01-01T00:00:00.000,<NA>,...,<NA>,-73.920247,40.754261,22.0,4307241.0,401.0,QN70,15900.0,4.006750e+09,POINT (-73.920246508724 40.754261466804)
1,41476556,POPEYES,Queens,165-25,LIBERTY AVENUE,11433,7185234233,Chicken,2025-02-28T00:00:00.000,Violations were cited in the following area(s).,...,Cycle Inspection / Initial Inspection,-73.791514,40.702224,27.0,4216231.0,412.0,QN61,44400.0,4.101550e+09,POINT (-73.79151424692 40.702223564768)
2,50088657,MINNIE'S BAR,Brooklyn,885,4 AVENUE,11232,7188324855,American,2024-10-02T00:00:00.000,Violations were cited in the following area(s).,...,Cycle Inspection / Initial Inspection,-74.002808,40.655845,38.0,3010178.0,307.0,BK32,8400.0,3.006850e+09,POINT (-74.002807564897 40.655845217215)
3,50047825,"L'AMICO, BACK BAR",Manhattan,839,6 AVENUE,10001,2122014065,American,2024-06-13T00:00:00.000,Violations were cited in the following area(s).,...,Cycle Inspection / Initial Inspection,-73.989906,40.746894,3.0,1015146.0,105.0,MN17,9500.0,1.008058e+09,POINT (-73.989905606376 40.746894352563)
4,50181761,DRAVIDA,Manhattan,211,1 AVENUE,10003,4016995421,Indian,2026-03-31T00:00:00.000,Violations were cited in the following area(s).,...,Pre-permit (Non-operational) / Initial Inspection,-73.983248,40.730420,2.0,1006491.0,103.0,MN22,4000.0,1.004540e+09,POINT (-73.983247537461 40.730419502848)


#### Perfilamiento inicial

In [7]:
df = df_raw

total_rows_raw = df.shape[0].compute()
total_columns = len(df.columns)

print("Filas originales:", total_rows_raw)
print("Columnas:", total_columns)

Filas originales: 296740
Columnas: 27


In [8]:
null_summary = df[
    ["camis", "inspection_date", "score", "boro", "critical_flag", "violation_code"]
].isna().sum().compute()

null_summary

camis                  0
inspection_date        0
score              17061
boro                   0
critical_flag          0
violation_code      5833
dtype: int64

In [9]:
df["boro"].value_counts(dropna=False).compute()

boro
0                   330
Manhattan        110189
Queens            73748
Bronx             27118
Brooklyn          75544
Staten Island      9811
Name: count, dtype: int64[pyarrow]

## Limpieza base

In [10]:
text_columns = [
    "camis", "dba", "boro", "building", "street", "zipcode",
    "phone", "cuisine_description", "action", "violation_code",
    "violation_description", "critical_flag", "grade",
    "inspection_type", "nta"
]

for c in text_columns:
    df[c] = df[c].astype("object").str.strip()

df["dba"] = df["dba"].str.upper()
df["grade"] = df["grade"].str.upper()

In [11]:
df["inspection_date"] = dd.to_datetime(
    df["inspection_date"],
    errors="coerce"
)

df["grade_date"] = dd.to_datetime(
    df["grade_date"],
    errors="coerce"
)

df["record_date"] = dd.to_datetime(
    df["record_date"],
    errors="coerce"
)

df["score"] = dd.to_numeric(df["score"], errors="coerce")

In [12]:
df_clean = df[
    df["camis"].notnull() &
    df["inspection_date"].notnull() &
    (df["inspection_date"] != pd.Timestamp("1900-01-01"))
]

#### Variables auxiliares

In [13]:
df_clean["year"] = df_clean["inspection_date"].dt.year
df_clean["month"] = df_clean["inspection_date"].dt.month

df_clean["season"] = "Otra temporada"
df_clean["season"] = df_clean["season"].mask(
    df_clean["month"].isin([6, 7, 8]),
    "Verano"
)
df_clean["season"] = df_clean["season"].mask(
    df_clean["month"].isin([12, 1, 2]),
    "Invierno"
)

df_clean["is_critical"] = (
    df_clean["critical_flag"]
    .str.lower()
    .eq("critical")
    .astype("int64")
)

df_clean["has_violation"] = df_clean["violation_code"].notnull().astype("int64")

df_clean["inspection_id"] = (
    df_clean["camis"].astype(str) + "_" +
    df_clean["inspection_date"].dt.strftime("%Y-%m-%d") + "_" +
    df_clean["inspection_type"].astype(str)
)

In [14]:
df_clean["risk_level"] = "Sin puntaje"

df_clean["risk_level"] = df_clean["risk_level"].mask(
    df_clean["score"].notnull() & (df_clean["score"] <= 13),
    "Bajo"
)

df_clean["risk_level"] = df_clean["risk_level"].mask(
    df_clean["score"].notnull() &
    (df_clean["score"] >= 14) &
    (df_clean["score"] <= 27),
    "Medio"
)

df_clean["risk_level"] = df_clean["risk_level"].mask(
    df_clean["score"].notnull() & (df_clean["score"] >= 28),
    "Alto"
)

In [15]:
df_clean["sanitary_status"] = "Requiere revisión"

df_clean["sanitary_status"] = df_clean["sanitary_status"].mask(
    (df_clean["grade"] == "A") & (df_clean["is_critical"] == 0),
    "Excelente"
)

df_clean["sanitary_status"] = df_clean["sanitary_status"].mask(
    (df_clean["grade"] == "A") & (df_clean["is_critical"] == 1),
    "Aceptable con alerta"
)

df_clean["sanitary_status"] = df_clean["sanitary_status"].mask(
    df_clean["grade"].isin(["B", "C"]),
    "Riesgo moderado"
)

#### Clasificación de violaciones y grupo de inspección

In [16]:
contamination_codes = [
    "03D", "04G", "04H", "04I", "06C", "04B", "04C",
    "04D", "04F", "05A", "10B", "06E", "10H"
]

pest_codes = ["04K", "04L", "04M", "04N"]

origin_codes = ["03A", "03B", "03I", "28-05", "03C", "03F", "03E"]

permit_codes = [
    "04A", "18-01", "18-02", "18-08", "18-10",
    "18-11", "18-12", "18-13", "18-14", "18-15", "18-25"
]

In [17]:
def add_dask_categories(pdf):
    pdf = pdf.copy()

    code = pdf["violation_code"]

    pdf["violation_category"] = np.select(
        [
            code.isin(contamination_codes),
            code.isin(pest_codes),
            code.isin(origin_codes),
            code.isin(permit_codes),
            code.isna()
        ],
        [
            "Contaminación de alimentos",
            "Plagas",
            "Origen no aprobado",
            "Permisos y documentación",
            "Sin violación"
        ],
        default="Otros"
    )

    inspection_type_norm = (
        pdf["inspection_type"]
        .fillna("")
        .astype(str)
        .str.lower()
    )

    pdf["inspection_group"] = np.select(
        [
            inspection_type_norm.str.contains("initial", na=False),
            inspection_type_norm.str.contains("re-inspection", na=False),
            inspection_type_norm.str.contains("reinspection", na=False),
            inspection_type_norm.str.contains("compliance", na=False),
            inspection_type_norm.str.contains("reopening", na=False),
            inspection_type_norm.str.contains("limited", na=False)
        ],
        [
            "Initial",
            "Re-inspection",
            "Re-inspection",
            "Compliance",
            "Reopening",
            "Limited"
        ],
        default="Other"
    )

    return pdf

In [18]:
meta_categories = df_clean._meta.copy()
meta_categories["violation_category"] = pd.Series(dtype="object")
meta_categories["inspection_group"] = pd.Series(dtype="object")

df_clean = df_clean.map_partitions(
    add_dask_categories,
    meta=meta_categories
)

#### Corrección de boro usando zipcode

In [19]:
zip_boro_pdf = (
    df_clean[
        df_clean["boro"].notnull() &
        (~df_clean["boro"].isin(["0", "Missing", "MISSING", "missing"])) &
        df_clean["zipcode"].notnull()
    ][["zipcode", "boro"]]
    .drop_duplicates()
    .compute(scheduler="single-threaded")
)

zip_boro_pdf = zip_boro_pdf.drop_duplicates(subset=["zipcode"])
zip_boro_dict = dict(zip(zip_boro_pdf["zipcode"], zip_boro_pdf["boro"]))

print("Relaciones zipcode-boro:", len(zip_boro_dict))

Relaciones zipcode-boro: 220


In [20]:
def correct_boro_partition(pdf, mapping):
    pdf = pdf.copy()

    pdf["boro_corrected"] = pdf["boro"]

    invalid_boro = (
        pdf["boro_corrected"].isna() |
        pdf["boro_corrected"].isin(["0", "Missing", "MISSING", "missing"])
    )

    pdf.loc[invalid_boro, "boro_corrected"] = (
        pdf.loc[invalid_boro, "zipcode"].map(mapping)
    )

    pdf["boro_corrected"] = pdf["boro_corrected"].fillna("Sin datos")

    return pdf

In [21]:
meta_boro = df_clean._meta.copy()
meta_boro["boro_corrected"] = pd.Series(dtype="object")

df_clean = df_clean.map_partitions(
    correct_boro_partition,
    zip_boro_dict,
    meta=meta_boro
)

# Base final para análisis

In [22]:
df_analysis = df_clean[
    df_clean["boro_corrected"].notnull() &
    (df_clean["boro_corrected"] != "Sin datos") &
    df_clean["score"].notnull() &
    df_clean["inspection_date"].notnull()
]

In [23]:
total_rows_clean = df_clean.shape[0].compute(scheduler="single-threaded")
total_rows_analysis = df_analysis.shape[0].compute(scheduler="single-threaded")

print("Filas originales:", total_rows_raw)
print("Filas después de limpieza base:", total_rows_clean)
print("Filas válidas para análisis:", total_rows_analysis)

Filas originales: 296740
Filas después de limpieza base: 293267
Filas válidas para análisis: 279493


In [24]:
df_analysis.head()

,camis,dba,boro,building,street,zipcode,phone,cuisine_description,inspection_date,action,...,month,season,is_critical,has_violation,inspection_id,risk_level,sanitary_status,violation_category,inspection_group,boro_corrected
1,41476556,POPEYES,Queens,165-25,LIBERTY AVENUE,11433,7185234233,Chicken,2025-02-28,Violations were cited in the following area(s).,...,2,Invierno,1,1,41476556_2025-02-28_Cycle Inspection / Initial...,Medio,Requiere revisión,Otros,Initial,Queens
2,50088657,MINNIE'S BAR,Brooklyn,885,4 AVENUE,11232,7188324855,American,2024-10-02,Violations were cited in the following area(s).,...,10,Otra temporada,1,1,50088657_2024-10-02_Cycle Inspection / Initial...,Medio,Requiere revisión,Plagas,Initial,Brooklyn
3,50047825,"L'AMICO, BACK BAR",Manhattan,839,6 AVENUE,10001,2122014065,American,2024-06-13,Violations were cited in the following area(s).,...,6,Verano,1,1,50047825_2024-06-13_Cycle Inspection / Initial...,Alto,Requiere revisión,Plagas,Initial,Manhattan
4,50181761,DRAVIDA,Manhattan,211,1 AVENUE,10003,4016995421,Indian,2026-03-31,Violations were cited in the following area(s).,...,3,Otra temporada,0,1,50181761_2026-03-31_Pre-permit (Non-operationa...,Alto,Requiere revisión,Otros,Initial,Manhattan
5,50132762,MIKADO,Queens,116-09,METROPOLITAN AVENUE,11418,7185590088,Japanese,2023-07-27,Violations were cited in the following area(s).,...,7,Verano,1,1,50132762_2023-07-27_Pre-permit (Operational) /...,Medio,Requiere revisión,Otros,Compliance,Queens


# Guardar base limpia en GCS por particiones

In [25]:
LOCAL_TMP_DIR = "/tmp/dask_processed_parts"
os.makedirs(LOCAL_TMP_DIR, exist_ok=True)

# Limpia destino si existe. Si no existe, no pasa nada.
subprocess.run(
    f"gcloud storage ls {GCS_PROCESSED_DASK} >/dev/null 2>&1 && "
    f"gcloud storage rm -r {GCS_PROCESSED_DASK} || true",
    shell=True
)

nparts = df_analysis.npartitions
print(f"Total de particiones a procesar: {nparts}")

for i in range(nparts):
    print(f"\nProcesando partición {i + 1}/{nparts}...")

    try:
        pdf_part = df_analysis.get_partition(i).compute(
            scheduler="single-threaded"
        )

        print(f"Filas en partición {i + 1}: {len(pdf_part)}")

        if len(pdf_part) > 0:
            local_file = f"{LOCAL_TMP_DIR}/part-{i:04d}.parquet"

            pdf_part.to_parquet(
                local_file,
                index=False,
                engine="pyarrow"
            )

            subprocess.run(
                f"gcloud storage cp {local_file} {GCS_PROCESSED_DASK}",
                shell=True,
                check=True
            )

            os.remove(local_file)

        del pdf_part
        gc.collect()

        print(f"Partición {i + 1}/{nparts} guardada correctamente.")

    except Exception as e:
        print(f"Error en partición {i + 1}: {e}")
        break

print("\nCheckpoint Dask guardado correctamente.")

Removing objects:
  
Removing gs://proyectobigdata2026/processed/dask/restaurant_inspections_clean/part-0000.parquet#1777998532378226...
Removing gs://proyectobigdata2026/processed/dask/restaurant_inspections_clean/part-0002.parquet#1777998545285700...
Removing gs://proyectobigdata2026/processed/dask/restaurant_inspections_clean/part-0001.parquet#1777998538894283...
Removing gs://proyectobigdata2026/processed/dask/restaurant_inspections_clean/part-0003.parquet#1777998551157795...



Total de particiones a procesar: 4

Procesando partición 1/4...
Filas en partición 1: 69890


Copying file:///tmp/dask_processed_parts/part-0000.parquet to gs://proyectobigdata2026/processed/dask/restaurant_inspections_clean/part-0000.parquet
  
...


Partición 1/4 guardada correctamente.

Procesando partición 2/4...
Filas en partición 2: 69804


Copying file:///tmp/dask_processed_parts/part-0001.parquet to gs://proyectobigdata2026/processed/dask/restaurant_inspections_clean/part-0001.parquet
  
...


Partición 2/4 guardada correctamente.

Procesando partición 3/4...
Filas en partición 3: 69962


Copying file:///tmp/dask_processed_parts/part-0002.parquet to gs://proyectobigdata2026/processed/dask/restaurant_inspections_clean/part-0002.parquet
  
...


Partición 3/4 guardada correctamente.

Procesando partición 4/4...
Filas en partición 4: 69837


Copying file:///tmp/dask_processed_parts/part-0003.parquet to gs://proyectobigdata2026/processed/dask/restaurant_inspections_clean/part-0003.parquet
  
...


Partición 4/4 guardada correctamente.

Checkpoint Dask guardado correctamente.


In [26]:
!gcloud storage ls gs://proyectobigdata2026/processed/dask/restaurant_inspections_clean_parts/

gs://proyectobigdata2026/processed/dask/restaurant_inspections_clean_parts/part-0000.parquet
gs://proyectobigdata2026/processed/dask/restaurant_inspections_clean_parts/part-0001.parquet


Desde este punto ya dejamos de trabajar con el grafo pesado que venía del CSV. Ahora trabajamos desde la base limpia.

In [27]:
df_analysis = dd.read_parquet(
    GCS_PROCESSED_DASK + "*.parquet",
    storage_options=GCS_STORAGE_OPTIONS
)

df_analysis.head()

,camis,dba,boro,building,street,zipcode,phone,cuisine_description,inspection_date,action,...,month,season,is_critical,has_violation,inspection_id,risk_level,sanitary_status,violation_category,inspection_group,boro_corrected
0,41476556,POPEYES,Queens,165-25,LIBERTY AVENUE,11433,7185234233,Chicken,2025-02-28,Violations were cited in the following area(s).,...,2,Invierno,1,1,41476556_2025-02-28_Cycle Inspection / Initial...,Medio,Requiere revisión,Otros,Initial,Queens
1,50088657,MINNIE'S BAR,Brooklyn,885,4 AVENUE,11232,7188324855,American,2024-10-02,Violations were cited in the following area(s).,...,10,Otra temporada,1,1,50088657_2024-10-02_Cycle Inspection / Initial...,Medio,Requiere revisión,Plagas,Initial,Brooklyn
2,50047825,"L'AMICO, BACK BAR",Manhattan,839,6 AVENUE,10001,2122014065,American,2024-06-13,Violations were cited in the following area(s).,...,6,Verano,1,1,50047825_2024-06-13_Cycle Inspection / Initial...,Alto,Requiere revisión,Plagas,Initial,Manhattan
3,50181761,DRAVIDA,Manhattan,211,1 AVENUE,10003,4016995421,Indian,2026-03-31,Violations were cited in the following area(s).,...,3,Otra temporada,0,1,50181761_2026-03-31_Pre-permit (Non-operationa...,Alto,Requiere revisión,Otros,Initial,Manhattan
4,50132762,MIKADO,Queens,116-09,METROPOLITAN AVENUE,11418,7185590088,Japanese,2023-07-27,Violations were cited in the following area(s).,...,7,Verano,1,1,50132762_2023-07-27_Pre-permit (Operational) /...,Medio,Requiere revisión,Otros,Compliance,Queens


#### Activar Client de Dask para CRUD y KPIs

In [28]:
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(
    n_workers=1,
    threads_per_worker=2,
    processes=False,
    memory_limit="5GB"
)

client = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://10.138.0.4:8787/status,
Dashboard: http://10.138.0.4:8787/status,Workers: 1
Total threads: 2,Total memory: 4.66 GiB
Status: running,Using processes: False
Comm: inproc://10.138.0.4/174013/1,Workers: 0
Dashboard: http://10.138.0.4:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: inproc://10.138.0.4/174013/4,Total threads: 2
Dashboard: http://10.138.0.4:45219/status,Memory: 4.66 GiB
Nanny: None,


2026-05-05 16:35:58,192 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle ed466379b15b4eb8ee870824f0f75373 initialized by task ('shuffle-transfer-ed466379b15b4eb8ee870824f0f75373', 1) executed on worker inproc://10.138.0.4/174013/4
2026-05-05 16:35:58,353 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle ed466379b15b4eb8ee870824f0f75373 deactivated due to stimulus 'task-finished-1777998958.350211'
2026-05-05 16:35:58,898 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 480f486addf34f7372982efb7e7dbef6 initialized by task ('shuffle-transfer-480f486addf34f7372982efb7e7dbef6', 1) executed on worker inproc://10.138.0.4/174013/4
2026-05-05 16:35:59,068 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 480f486addf34f7372982efb7e7dbef6 deactivated due to stimulus 'task-finished-1777998959.0675995'
2026-05-05 16:36:02,055 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 744822523bcf398e9f5199fca5fe9b34 initialized by task ('shuffle-transfer-7

# CRUD con Dask

## DASK-C1: Create — categoría de violación

In [29]:
crud_c1_violation_category = (
    df_analysis.groupby("violation_category")
    .size()
    .compute()
    .reset_index(name="total_records")
    .sort_values("total_records", ascending=False)
)

crud_c1_violation_category

,violation_category,total_records
2,Otros,178835
0,Contaminación de alimentos,53906
4,Plagas,34698
3,Permisos y documentación,8113
1,Origen no aprobado,2387
5,Sin violación,1554


## DASK-C2: Create — grupo de inspección

In [30]:
crud_c2_inspection_group = (
    df_analysis.groupby("inspection_group")
    .size()
    .compute()
    .reset_index(name="total_records")
    .sort_values("total_records", ascending=False)
)

crud_c2_inspection_group

,inspection_group,total_records
1,Initial,202232
2,Re-inspection,70166
0,Compliance,3625
3,Reopening,3470


## DASK-R1: Read — score promedio por año

In [31]:
crud_r1_score_by_year = (
    df_analysis.groupby("year")["score"]
    .mean()
    .compute()
    .reset_index()
)

crud_r1_score_by_year.columns = ["year", "avg_score"]
crud_r1_score_by_year = crud_r1_score_by_year.sort_values("year")

crud_r1_score_by_year

,year,avg_score
0,2007,30.468085
1,2008,38.909091
2,2009,20.0
3,2010,30.090909
4,2011,16.0
5,2012,15.333333
6,2013,4.0
7,2014,27.0
8,2015,16.5
9,2016,15.577075


## DASK-R2: Read — distribución porcentual de calificaciones

In [32]:
grade_counts = df_analysis["grade"].value_counts(dropna=False).compute()

crud_r2_grade_distribution = (
    (grade_counts / grade_counts.sum()) * 100
).round(2).reset_index()

crud_r2_grade_distribution.columns = ["grade", "percentage"]

crud_r2_grade_distribution

,grade,percentage
0,A,35.49
1,P,0.31
2,Z,1.54
3,NaN,47.75
4,B,6.64
5,C,4.85
6,N,3.41


## DASK-U1: Update — comparación de boro antes y después

In [33]:
boro_after = (
    df_analysis["boro_corrected"]
    .value_counts(dropna=False)
    .compute()
    .reset_index()
)

boro_after.columns = ["boro_corrected", "total_records"]

boro_after

,boro_corrected,total_records
0,Bronx,25801
1,Brooklyn,71621
2,Staten Island,9510
3,Manhattan,103046
4,Queens,69515


## DASK-D1: Delete lógico — resumen de filtrado

In [34]:
crud_d1_delete_summary = pd.DataFrame({
    "stage": [
        "dataset_original",
        "dataset_clean",
        "dataset_analysis"
    ],
    "total_rows": [
        total_rows_raw,
        total_rows_clean,
        total_rows_analysis
    ]
})

crud_d1_delete_summary["rows_removed_vs_original"] = (
    total_rows_raw - crud_d1_delete_summary["total_rows"]
)

crud_d1_delete_summary

,stage,total_rows,rows_removed_vs_original
0,dataset_original,296740,0
1,dataset_clean,293267,3473
2,dataset_analysis,279493,17247


# KPIs con Dask

## KPI 3: Reincidencia post-clausura

In [35]:
df_closures = df_analysis[
    df_analysis["action"]
    .fillna("")
    .str.lower()
    .str.contains("establishment closed by dohmh", na=False)
]

closure_dates = (
    df_closures.groupby("camis")["inspection_date"]
    .min()
    .reset_index()
)

closure_dates = closure_dates.rename(columns={
    "inspection_date": "closure_date"
})

df_post_closure = df_analysis.merge(
    closure_dates,
    on="camis",
    how="inner"
)

df_post_closure = df_post_closure[
    df_post_closure["inspection_date"] > df_post_closure["closure_date"]
]

post_closure_violations = (
    df_post_closure.groupby("camis")["violation_code"]
    .count()
    .compute()
)

post_closure_inspections = (
    df_post_closure[["camis", "inspection_id"]]
    .drop_duplicates()
    .groupby("camis")
    .size()
    .compute()
)

restaurant_catalog = (
    df_analysis.groupby("camis")
    .agg({
        "dba": "first",
        "boro_corrected": "first",
        "zipcode": "first"
    })
    .reset_index()
    .compute()
)

In [36]:
kpi3_reincidencia_post_clausura = pd.concat(
    [
        post_closure_violations.rename("post_closure_violations"),
        post_closure_inspections.rename("post_closure_inspections")
    ],
    axis=1
).reset_index()

kpi3_reincidencia_post_clausura = kpi3_reincidencia_post_clausura.merge(
    restaurant_catalog,
    on="camis",
    how="left"
)

kpi3_reincidencia_post_clausura = (
    kpi3_reincidencia_post_clausura
    .sort_values("post_closure_violations", ascending=False)
)

kpi3_reincidencia_post_clausura.head(10)

,camis,post_closure_violations,post_closure_inspections,dba,boro_corrected,zipcode
836,50138270,67,11,MEEM SPICY GROCERY AND DELI,Manhattan,10016
620,50111296,55,10,BIG WONG,Manhattan,10013
194,41696014,52,9,NOISETTE,Queens,11103
244,50009411,50,8,DELICIAS CALENAS #4,Queens,11435
751,50130108,46,8,CENTRAL PARK CAFE,Manhattan,10019
164,41628099,44,10,LLOYD'S CARROT CAKE,Manhattan,10029
752,50130345,43,7,AFFY'S GRILL,Queens,11378
718,50124605,43,6,CHEF TAN,Manhattan,10003
843,50139259,41,6,SHANGHAI TIME,Manhattan,10018
323,50048617,41,10,HALAL BUFFET,Queens,11435


## KPI 4: Tasa de violaciones por tipo

In [37]:
df_violations = df_analysis[df_analysis["violation_code"].notnull()]

violation_counts = (
    df_violations.groupby("violation_category")
    .size()
    .compute()
)

kpi4_tasa_violaciones_tipo = (
    (violation_counts / violation_counts.sum()) * 100
).round(2).reset_index()

kpi4_tasa_violaciones_tipo.columns = [
    "violation_category",
    "violation_rate_percentage"
]

kpi4_tasa_violaciones_tipo = (
    kpi4_tasa_violaciones_tipo
    .sort_values("violation_rate_percentage", ascending=False)
)

kpi4_tasa_violaciones_tipo

,violation_category,violation_rate_percentage
2,Otros,64.34
0,Contaminación de alimentos,19.39
4,Plagas,12.48
3,Permisos y documentación,2.92
1,Origen no aprobado,0.86


## KPI 7: Tiempo promedio de subsanación

In [38]:
inspection_grade = (
    df_analysis[
        ["camis", "dba", "inspection_date", "grade"]
    ]
    .dropna(subset=["grade"])
    .drop_duplicates()
    .compute()
)

inspection_grade["grade"] = inspection_grade["grade"].astype(str).str.upper()

In [39]:
df_bad = inspection_grade[
    inspection_grade["grade"].isin(["B", "C"])
][["camis", "dba", "inspection_date", "grade"]].copy()

df_bad = df_bad.rename(columns={
    "inspection_date": "fecha_caida",
    "grade": "grade_caida"
})

df_good = inspection_grade[
    inspection_grade["grade"] == "A"
][["camis", "inspection_date"]].copy()

df_good = df_good.rename(columns={
    "inspection_date": "fecha_subsanacion"
})

df_recovery = df_bad.merge(
    df_good,
    on="camis",
    how="inner"
)

df_recovery = df_recovery[
    df_recovery["fecha_subsanacion"] > df_recovery["fecha_caida"]
]

if len(df_recovery) > 0:
    df_tiempos = (
        df_recovery
        .groupby(["camis", "dba", "fecha_caida", "grade_caida"], dropna=False)
        ["fecha_subsanacion"]
        .min()
        .reset_index()
    )

    df_tiempos["dias_subsanacion"] = (
        df_tiempos["fecha_subsanacion"] - df_tiempos["fecha_caida"]
    ).dt.days

    kpi7_tiempo_promedio_subsanacion = pd.DataFrame({
        "metric": ["tiempo_promedio_subsanacion_dias"],
        "value": [round(df_tiempos["dias_subsanacion"].mean(), 2)]
    })

    kpi7_top_restaurantes = (
        df_tiempos
        .groupby(["camis", "dba"], dropna=False)["dias_subsanacion"]
        .mean()
        .reset_index()
        .sort_values("dias_subsanacion", ascending=False)
        .head(10)
    )
else:
    df_tiempos = pd.DataFrame()
    kpi7_tiempo_promedio_subsanacion = pd.DataFrame({
        "metric": ["tiempo_promedio_subsanacion_dias"],
        "value": [np.nan]
    })
    kpi7_top_restaurantes = pd.DataFrame()

In [40]:
kpi7_tiempo_promedio_subsanacion

,metric,value
0,tiempo_promedio_subsanacion_dias,444.88


In [41]:
kpi7_top_restaurantes

,camis,dba,dias_subsanacion
619,41619055,SONS OF ESSEX,1091.0
1289,50061281,JOE & PAT'S,1085.0
35,40385885,HUNTER'S STEAK & ALE HOUSE,1078.0
1836,50098711,PHO BEST III,1071.0
454,41418789,THE ORIGINAL EMILIO'S PIZZA,1071.0
25,40379149,MONTE'S TRATTORIA,1065.0
1618,50086215,LARUSTICA PIZZA,1064.0
622,41621699,MI PASO CENTRO AMERICANO RESTAURANT,1060.0
1837,50098945,DIOS PROVEEDOR BAKERY,1058.0
495,41476400,CHOP CHOP,1058.0


# Guardado de resultados pequeños en GCS

Para evitar problemas de pandas.to_csv directo a GCS, usamos una función segura: guarda local y luego sube con gcloud storage cp.

In [42]:
def save_pandas_csv_to_gcs(pdf, gcs_path, index=False):
    local_tmp = "/tmp/" + os.path.basename(gcs_path)

    pdf.to_csv(local_tmp, index=index)

    subprocess.run(
        f"gcloud storage cp {local_tmp} {gcs_path}",
        shell=True,
        check=True
    )

    os.remove(local_tmp)
    print(f"Guardado en: {gcs_path}")

In [43]:
save_pandas_csv_to_gcs(
    crud_c1_violation_category,
    GCS_OUTPUTS_DASK + "dask_crud_c1_violation_category.csv"
)

save_pandas_csv_to_gcs(
    crud_c2_inspection_group,
    GCS_OUTPUTS_DASK + "dask_crud_c2_inspection_group.csv"
)

save_pandas_csv_to_gcs(
    crud_r1_score_by_year,
    GCS_OUTPUTS_DASK + "dask_crud_r1_score_by_year.csv"
)

save_pandas_csv_to_gcs(
    crud_r2_grade_distribution,
    GCS_OUTPUTS_DASK + "dask_crud_r2_grade_distribution.csv"
)

save_pandas_csv_to_gcs(
    boro_after,
    GCS_OUTPUTS_DASK + "dask_crud_u1_boro_corrected_distribution.csv"
)

save_pandas_csv_to_gcs(
    crud_d1_delete_summary,
    GCS_OUTPUTS_DASK + "dask_crud_d1_delete_summary.csv"
)

Copying file:///tmp/dask_crud_c1_violation_category.csv to gs://proyectobigdata2026/results/outputs/dask/dask_crud_c1_violation_category.csv
  
.


Guardado en: gs://proyectobigdata2026/results/outputs/dask/dask_crud_c1_violation_category.csv


Copying file:///tmp/dask_crud_c2_inspection_group.csv to gs://proyectobigdata2026/results/outputs/dask/dask_crud_c2_inspection_group.csv
  
.


Guardado en: gs://proyectobigdata2026/results/outputs/dask/dask_crud_c2_inspection_group.csv


Copying file:///tmp/dask_crud_r1_score_by_year.csv to gs://proyectobigdata2026/results/outputs/dask/dask_crud_r1_score_by_year.csv
  
.


...

Guardado en: gs://proyectobigdata2026/results/outputs/dask/dask_crud_r2_grade_distribution.csv


Copying file:///tmp/dask_crud_u1_boro_corrected_distribution.csv to gs://proyectobigdata2026/results/outputs/dask/dask_crud_u1_boro_corrected_distribution.csv
  
.


Guardado en: gs://proyectobigdata2026/results/outputs/dask/dask_crud_u1_boro_corrected_distribution.csv


Copying file:///tmp/dask_crud_d1_delete_summary.csv to gs://proyectobigdata2026/results/outputs/dask/dask_crud_d1_delete_summary.csv
  
.


Guardado en: gs://proyectobigdata2026/results/outputs/dask/dask_crud_d1_delete_summary.csv


In [44]:
save_pandas_csv_to_gcs(
    kpi3_reincidencia_post_clausura,
    GCS_KPIS_DASK + "dask_kpi3_reincidencia_post_clausura.csv"
)

save_pandas_csv_to_gcs(
    kpi4_tasa_violaciones_tipo,
    GCS_KPIS_DASK + "dask_kpi4_tasa_violaciones_tipo.csv"
)

save_pandas_csv_to_gcs(
    kpi7_tiempo_promedio_subsanacion,
    GCS_KPIS_DASK + "dask_kpi7_tiempo_promedio_subsanacion.csv"
)

save_pandas_csv_to_gcs(
    kpi7_top_restaurantes,
    GCS_KPIS_DASK + "dask_kpi7_top_restaurantes_subsanacion.csv"
)

Copying file:///tmp/dask_kpi3_reincidencia_post_clausura.csv to gs://proyectobigdata2026/results/kpis/dask/dask_kpi3_reincidencia_post_clausura.csv
  
.


Guardado en: gs://proyectobigdata2026/results/kpis/dask/dask_kpi3_reincidencia_post_clausura.csv


Copying file:///tmp/dask_kpi4_tasa_violaciones_tipo.csv to gs://proyectobigdata2026/results/kpis/dask/dask_kpi4_tasa_violaciones_tipo.csv
  
.


Guardado en: gs://proyectobigdata2026/results/kpis/dask/dask_kpi4_tasa_violaciones_tipo.csv


Copying file:///tmp/dask_kpi7_tiempo_promedio_subsanacion.csv to gs://proyectobigdata2026/results/kpis/dask/dask_kpi7_tiempo_promedio_subsanacion.csv
  
.


Guardado en: gs://proyectobigdata2026/results/kpis/dask/dask_kpi7_tiempo_promedio_subsanacion.csv


Copying file:///tmp/dask_kpi7_top_restaurantes_subsanacion.csv to gs://proyectobigdata2026/results/kpis/dask/dask_kpi7_top_restaurantes_subsanacion.csv
  
.


Guardado en: gs://proyectobigdata2026/results/kpis/dask/dask_kpi7_top_restaurantes_subsanacion.csv
